In [ ]:
# Select empty non-cartoon Silver rows and read only their archived HTML files.
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, IntegerType, StringType, StructField, StructType

articles = spark.table('`news-analytics`.silver.articles')
missing = (articles
    .filter(F.trim(F.col('body_text')) == '')
    .filter(~F.coalesce(F.array_contains(F.col('categories'), 'Cartoons'), F.lit(False)))
    .select('canonical_url', 'source_path', 'categories'))
paths = [row.source_path for row in missing.select('source_path').distinct().collect()]
if not paths:
    raise ValueError('No empty non-cartoon article bodies to profile')
files = spark.read.format('binaryFile').load(paths).select('path', 'content')
print('FILES TO PROFILE:', len(paths))


In [ ]:
# Describe each page's content layout, text paths, and media without changing Silver.
profile_schema = StructType([
    StructField('content_found', BooleanType()),
    StructField('top_level_layout', StringType()),
    StructField('dominant_text_path', StringType()),
    StructField('text_elements', IntegerType()),
    StructField('text_element_chars', IntegerType()),
    StructField('content_chars', IntegerType()),
    StructField('images', IntegerType()),
    StructField('embeds', IntegerType()),
    StructField('sample_text', StringType()),
])

def profile_html(content):
    from collections import defaultdict
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(content.decode('utf-8', errors='replace'), 'html.parser')
    root = soup.select_one('.jeg_custom_content_wrapper .entry-content .content-inner')
    if root is None:
        return {'content_found': False, 'top_level_layout': 'none',
                'dominant_text_path': 'none', 'text_elements': 0,
                'text_element_chars': 0, 'content_chars': 0,
                'images': 0, 'embeds': 0, 'sample_text': ''}

    # Keep stable tag and class names so pages with the same layout group together.
    def tag_name(node):
        classes = [name for name in node.get('class', [])
                   if not name.startswith(('vc_custom_', 'wp-image-'))]
        return node.name + ''.join('.' + name for name in classes[:2])

    top_nodes = [child for child in root.children if getattr(child, 'name', None)]
    top_layout = ' > '.join(list(dict.fromkeys(tag_name(node) for node in top_nodes))[:4]) or 'none'
    chars_by_path = defaultdict(int)
    text_nodes = []
    for node in root.select('p, h2, h3, li'):
        value = node.get_text(' ', strip=True)
        if not value:
            continue
        ancestors = []
        parent = node.parent
        while parent is not root:
            ancestors.append(tag_name(parent))
            parent = parent.parent
        path = ' > '.join([*reversed(ancestors), node.name])
        chars_by_path[path] += len(value)
        text_nodes.append(value)
    dominant_path = max(chars_by_path, key=chars_by_path.get) if chars_by_path else 'none'
    return {
        'content_found': True,
        'top_level_layout': top_layout,
        'dominant_text_path': dominant_path,
        'text_elements': len(text_nodes),
        'text_element_chars': sum(map(len, text_nodes)),
        'content_chars': len(root.get_text(' ', strip=True)),
        'images': len(root.find_all('img')),
        'embeds': len(root.select('iframe, video, .twitter-tweet, .instagram-media, .tiktok-embed')),
        'sample_text': max(text_nodes, key=len)[:160] if text_nodes else '',
    }

profile_udf = F.udf(profile_html, profile_schema)
profiled_files = files.select(F.col('path').alias('source_path'), profile_udf('content').alias('profile'))
profiled = missing.join(profiled_files, 'source_path').select('canonical_url', 'source_path', 'categories', 'profile.*').cache()
print('PROFILED ARTICLES:', profiled.count())


In [ ]:
# Rank repeated layouts and show example URLs, then inspect individual file profiles.
groups = (profiled
    .groupBy('content_found', 'top_level_layout', 'dominant_text_path')
    .agg(F.count('*').alias('files'),
         F.sum((F.col('text_element_chars') > 0).cast('int')).alias('files_with_text'),
         F.max('text_element_chars').alias('max_text_chars'),
         F.max('images').alias('max_images'),
         F.max('embeds').alias('max_embeds'),
         F.slice(F.sort_array(F.collect_set('canonical_url')), 1, 3).alias('example_urls'))
    .orderBy(F.desc('files'), 'top_level_layout', 'dominant_text_path'))
display(groups)
display(profiled.select('canonical_url', 'source_path', 'categories',
    'top_level_layout', 'dominant_text_path', 'text_elements',
    'text_element_chars', 'content_chars', 'images', 'embeds', 'sample_text')
    .orderBy('top_level_layout', 'dominant_text_path', 'canonical_url'))
